This notebook serves as a demonstration of the LASP SDTP application.

In [66]:
# Some commands to make things easier to test within this notebook
from lasp_sdtp.database.database_controller import db
from tests import conftest
# db.session.query(db.FileQueue).delete()
# db.session.query(db.Transactions).delete()
# db.session.query(db.Accounts).delete()
# db.session.query(db.FileMetadata).delete()
db.engine.execute('TRUNCATE TABLE TRANSACTIONS DROP STORAGE')
db.engine.execute('TRUNCATE TABLE FILE_METADATA DROP STORAGE')
db.engine.execute('ALTER TABLE file_metadata ADD (foo VARCHAR2(1));')
db.session.commit()
conftest._add_file_metadata_entries()

DatabaseError: (cx_Oracle.DatabaseError) ORA-01735: invalid ALTER TABLE option
[SQL: ALTER TABLE file_metadata ADD (foo VARCHAR2(1));]
(Background on this error at: https://sqlalche.me/e/14/4xp6)

# Configuration

In [2]:
from lasp_sdtp.config import subscriber_config
subscriber_config

{'account_expiration_period': 1800,
 'checksum_type': 'sha256',
 'distinguished_name': '/C=US/ST=Maryland/L=Greenbelt/O=NASA/OU=Goddard Earth Science Data and Information Services Center/CN=GES DISC/emailAddress=guang-dih.lei@nasa.gov',
 'expiration_period': 180,
 'max_num_files': 10000,
 'num_download_threads': 5,
 'username': 'ges_disc'}

In [3]:
subscriber_config['account_expiration_period'] = 1800
subscriber_config['checksum_type'] = 'sha256'
subscriber_config['distinguished_name'] = ''
subscriber_config['expiration_period'] = 180
subscriber_config['max_num_files'] = 10000
subscriber_config['num_download_threads'] = 5
subscriber_config['username'] = 'ges_disc'

In [4]:
subscriber_config

{'account_expiration_period': 1800,
 'checksum_type': 'sha256',
 'distinguished_name': '',
 'expiration_period': 180,
 'max_num_files': 10000,
 'num_download_threads': 5,
 'username': 'ges_disc'}

# Register Account

Register a new account.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned.
    
Command: `curl PUT "http://127.0.0.1:8000/register -H "Cert-UID: <certificate>"`

In [5]:
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

# List Available Files

Return a list of available files given user-supplied parameters.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with a list of the available files and their metadata.  If no parameters are supplied, then all available files are returned.

Command: `curl GET "http://127.0.0.1:8000/files?param1=value1&param2=value2... -H "Cert-UID: <certificate>"`

#### Supported parameters:
- `stream` (e.g. `prod`)
- `ShortName` (e.g. `TSIS_SC_L2`)
- `version` (e.g. `v01`)
- `date` (e.g.`2022-01-01`)
- `start_date` (e.g. `2022-01-01`)
- `end_date` (e.g. `2022-01-02`)


Note that `date` can only be supplied if `start_date` and/or `end_date` are *not* supplied.  Also, if `start_date` is supplied, then `end_date` *must* be supplied, or vice-versa 

### Examples

In [6]:
# List all available files
! curl -i -X GET "http://127.0.0.1:8000/files" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:rz1ccsz6btvwky9ukk39ej7xhdstw9tbmrjoimd1xidt5dvnqnoh5haoi81ewtky", 
      "date": "2021-12-31", 
      "expires": "2023-01-15", 
      "fileid": 5413, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:ikoftkqzw0a9u3bbllwa9ilyg5tukxxv8px5lw95q32x5d4uxj030vu8ctw2lpt3", 
      "date": "2022-01-01", 
      "expires": "2023-01-15", 
      "fileid": 5414, 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:k55fi1rukesbnymcvdiu0911kszfcgszq9fddzgthknwga5vkbgcr2phcxqlm8h8", 
      "date": "2022-01-02", 
      "expires": "2023-01-15", 
      "fileid": 5415, 
      "name": "tsis2_sc_L2_v01_20220104_20220105.zip", 
      "shortname": "TSIS_SC_L2", 
  

In [7]:
# List all available files with ShortName of 'TSIS_SC_L2'
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:rz1ccsz6btvwky9ukk39ej7xhdstw9tbmrjoimd1xidt5dvnqnoh5haoi81ewtky", 
      "date": "2021-12-31", 
      "expires": "2023-01-15", 
      "fileid": 5413, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:ikoftkqzw0a9u3bbllwa9ilyg5tukxxv8px5lw95q32x5d4uxj030vu8ctw2lpt3", 
      "date": "2022-01-01", 
      "expires": "2023-01-15", 
      "fileid": 5414, 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:k55fi1rukesbnymcvdiu0911kszfcgszq9fddzgthknwga5vkbgcr2phcxqlm8h8", 
      "date": "2022-01-02", 
      "expires": "2023-01-15", 
      "fileid": 5415, 
      "name": "tsis2_sc_L2_v01_20220104_20220105.zip", 
      "shortname": "TSIS_SC_L2", 
  

In [8]:
# List all available files with specific ShortName and for specific date
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&date=2022-01-02" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:k55fi1rukesbnymcvdiu0911kszfcgszq9fddzgthknwga5vkbgcr2phcxqlm8h8", 
      "date": "2022-01-02", 
      "expires": "2023-01-15", 
      "fileid": 5415, 
      "name": "tsis2_sc_L2_v01_20220104_20220105.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }
  ]
}


In [9]:
# List all available files between range between dates
! curl -i -X GET "http://127.0.0.1:8000/files?start_date=2021-12-31&end_date=2022-01-05" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:rz1ccsz6btvwky9ukk39ej7xhdstw9tbmrjoimd1xidt5dvnqnoh5haoi81ewtky", 
      "date": "2021-12-31", 
      "expires": "2023-01-15", 
      "fileid": 5413, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:ikoftkqzw0a9u3bbllwa9ilyg5tukxxv8px5lw95q32x5d4uxj030vu8ctw2lpt3", 
      "date": "2022-01-01", 
      "expires": "2023-01-15", 
      "fileid": 5414, 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:k55fi1rukesbnymcvdiu0911kszfcgszq9fddzgthknwga5vkbgcr2phcxqlm8h8", 
      "date": "2022-01-02", 
      "expires": "2023-01-15", 
      "fileid": 5415, 
      "name": "tsis2_sc_L2_v01_20220104_20220105.zip", 
      "shortname": "TSIS_SC_L2", 
  

# Get Specific File

Return the contents of a file for a given `fileid`.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with the content of the file.  

Command: `curl GET "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [11]:
! curl -i -X GET "http://127.0.0.1:8000/files/5414" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220103_20220104.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220103_20220104.zip"
}


# Delete Specific File

Remove a specific file from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [13]:
! curl -i -X DELETE "http://127.0.0.1:8000/files/5414" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

# Delete Range of Files

Remove a range of files from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid_start>-<fileid_end> -H "Cert-UID: <certificate>"`

In [14]:
# First get a few files so that they are in the queue
! curl -i -X GET "http://127.0.0.1:8000/files/5413" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/5414" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/5415" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 142
SDTP-TransactionID: 2252
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 19 Jul 2022 16:07:22 GMT

{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220102_20220103.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220102_20220103.zip"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 142
SDTP-TransactionID: 2253
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 19 Jul 2022 16:07:22 GMT

{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220103_20220104.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220103_20220104.zip"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 142
SDTP-TransactionID: 2254
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Tue, 19 Jul 2022 16:07:22 GMT

{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220104_20220105.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220104_20220105.zip"
}


In [15]:
# Delete the range of files
! curl -i -X DELETE "http://127.0.0.1:8000/files/5413-5415" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"